## Stanza zu SQLite-DB

In [6]:
import os
import time
import sqlite3
import unicodedata
from stanza.utils.conll import CoNLL

def remove_polytonic_greek(text: str) -> str:
    """
    Removes diacritical marks from polytonic Greek, returning only base letters.
    """
    normalized_text = unicodedata.normalize('NFD', text)
    return ''.join(c for c in normalized_text if not unicodedata.combining(c) and c not in "῾᾿")

def convert_conllu_to_sqlite(input_path, conn):
    """
    Reads a CoNLL-U formatted file and stores the data in an SQLite database.
    """
    
    cursor = conn.cursor()
    
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS conllu_data (
        id INTEGER,
        text TEXT,
        lemma TEXT,
        upos TEXT,
        xpos TEXT,
        feats TEXT,
        head INTEGER,
        deprel TEXT,
        deps TEXT,
        misc TEXT
    )
    ''')
    
    with open(input_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    data = []
    
    for line in lines:
        line = line.strip()
        if line and not line.startswith("#"):
            columns = line.split("\t")
            if len(columns) == 10:
                word_id = int(columns[0])
                word_text = columns[1]  # Token
                lemma = columns[2]  # Lemma
                upos = columns[3]  # Universal POS Tag
                xpos = columns[4]  # Language-specific POS tag
                feats = columns[5]  # Morphological features
                head = int(columns[6]) if columns[6].isdigit() else None
                deprel = columns[7]  # Dependency relation
                deps = columns[8]  # Enhanced dependencies
                misc = columns[9]  # Miscellaneous
                
                word_text = remove_polytonic_greek(word_text)
                lemma = remove_polytonic_greek(lemma)
                
                data.append((word_id, word_text, lemma, upos, xpos, feats, head, deprel, deps, misc))
    
    cursor.executemany('''
    INSERT INTO conllu_data (id, text, lemma, upos, xpos, feats, head, deprel, deps, misc)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''', data)
    conn.commit()
    
    
    

def process_directory(input_dir, output_db):
    """
    Iterates through the directory and converts all CoNLL-U files to SQLite database format.
    """
    os.makedirs(os.path.dirname(output_db), exist_ok=True) 
    conn = sqlite3.connect(output_db)
    
    
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".conllu"):
                input_path = os.path.join(root, file)
                relative_path = os.path.relpath(input_path, os.path.join(input_dir, 'conllu'))
                
                
                
                
                start_time = time.time()
                convert_conllu_to_sqlite(input_path, conn)
                end_time = time.time()
                print(f"File '{input_path}' processed and added to database in {end_time - start_time:.2f} seconds.")

# Main execution
if __name__ == "__main__":
    input_directory = "./test-grc-out"
    output_database = os.path.join(input_directory, 'conllu_data.db')
    
    process_directory(input_directory, output_database)


File './test-grc-out\conllu\txt\chapter06.conllu' processed and added to database in 0.04 seconds.
File './test-grc-out\conllu\txt\chapter07.conllu' processed and added to database in 0.03 seconds.
